# 02b · 푸아송 과정의 세 정의와 순서통계량 (Three Definitions of the Poisson Process and the Order-Statistics Property)

| 항목 | 내용 |
|---|---|
| 교재 지도 | Durrett 3e §2.2 (간격 정의 → 독립·정상 증분, Thm 2.1) · Durrett 3e §2.4 (조건부 균등 순서통계량) · Ross 12e §5.3 (5.3.1 계수과정, 5.3.2 정의와 동치, 5.3.3 간격·대기시간, 5.3.5 도착시각의 조건부 분포) · G&S 4e §6.8 (엄밀한 동치 증명) |
| 선수 노트북 | 02a (지수분포·무기억성·Gamma 합), 00c (조건부 기대와 tower) |
| 예상 소요 | 90분 |
| 상태 | draft |

이 노트북의 단 하나의 아이디어: **푸아송 과정은 '계수(증분)', '간격', '무한소' 세 눈높이에서 같은 대상이고, 도착 수만 알려 주면 도착 시각은 완전히 무작위(균등 순서통계량)이다.** 그래서 같은 과정을 두 가지 방법(지수 간격 누적 / 개수 뽑고 균등점 정렬)으로 만들 수 있다.

## 0. 준비 (Setup)

In [ ]:
import os
import math
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Markdown
from scipy import stats
from spkit import rng_for, fast, scale, setup_plots
from spkit.mc import mc_estimate, mc_proportion, compare_table, assert_close
from spkit.poisson import poisson_arrivals, poisson_arrivals_conditional, count_in_bins
from spkit.renewal import renewal_times, renewal_count
from spkit.plots import plot_hist_vs_pmf, plot_hist_vs_pdf

SEED, rng = rng_for("02b")
setup_plots()
N_PATHS = scale(50_000)   # FAST(SPKIT_FAST=1) 모드에서는 1/10 크기
LAM, T = 2.0, 10.0        # 이 노트북 전체의 비율과 지평
print(f"SEED={SEED}, fast={fast()}, N_PATHS={N_PATHS}")

## 1. Recall — 지난 노트북에서

책을 덮고 먼저 답을 적어 본 뒤 정답을 펼쳐 보세요.

**Q1.** (02a) $X\sim\mathrm{Exp}(\lambda)$일 때 $P(X>s+t\mid X>s)$는?

<details><summary>정답</summary>

$e^{-\lambda t}=P(X>t)$. 무기억성(memorylessness)이다. 위험률(hazard rate)이 상수 $\lambda$라는 말과 같다.

</details>

**Q2.** (02a) 독립 $\mathrm{Exp}(\lambda_1),\dots,\mathrm{Exp}(\lambda_n)$의 최솟값 분포와 $i$번째가 최솟값일 확률은? 둘 사이 관계는?

<details><summary>정답</summary>

최솟값은 $\mathrm{Exp}(\sum_j\lambda_j)$, $i$번째가 최솟값일 확률은 $\lambda_i/\sum_j\lambda_j$, 그리고 두 확률변수(최솟값, argmin)는 서로 **독립**이다.

</details>

**Q3.** (02a) iid $\mathrm{Exp}(\lambda)$ $n$개의 합의 분포, 평균, 분산은? 꼬리확률은?

<details><summary>정답</summary>

$\mathrm{Gamma}(n,\lambda)$ (Erlang), 평균 $n/\lambda$, 분산 $n/\lambda^2$. 꼬리 $P(S_n>t)=\sum_{k<n}e^{-\lambda t}(\lambda t)^k/k!$ — 오늘 정리 1의 핵심 재료다.

</details>

**Q4 (연결 고리).** (01g) 자손 분포 $(p_0,p_1,p_2)=(1/4,1/4,1/2)$인 골턴-왓슨 과정의 소멸확률은?

<details><summary>정답</summary>

$m=1/4+1=5/4>1$이므로 소멸확률은 1보다 작다. $\varphi(s)=\tfrac14+\tfrac s4+\tfrac{s^2}2=s\iff 2s^2-3s+1=(2s-1)(s-1)=0$. 가장 작은 해 $q=1/2$.

</details>

## 2. 이론 — 내 말로 다시 쓰기

### 2.1 정의

- **계수과정(counting process)** $\{N(t),t\ge0\}$: $N(0)=0$, 정수값, 비감소, 우연속(right-continuous). $N(t)-N(s)$는 구간 $(s,t]$의 도착 수.
- **정의 1 (계수/증분 정의).** 비율 $\lambda$ 푸아송 과정은 (i) $N(0)=0$, (ii) 서로소 구간의 증분들이 독립(independent increments), (iii) $N(t+s)-N(s)\sim\mathrm{Poisson}(\lambda t)$ (정상 증분, stationary increments).
- **정의 2 (간격 정의).** iid $\tau_i\sim\mathrm{Exp}(\lambda)$, 도착시각 $T_n=\tau_1+\cdots+\tau_n$, $N(t)=\max\{n:T_n\le t\}$.
- **정의 3 (무한소 정의).** (i) $N(0)=0$, (ii) 독립·정상 증분, (iii) $P(N(h)=1)=\lambda h+o(h)$, (iv) $P(N(h)\ge2)=o(h)$.
- **순서통계량(order statistics).** $U_1,\dots,U_n$ iid $U(0,T)$를 오름차순으로 정렬한 $U_{(1)}<\cdots<U_{(n)}$. 결합밀도는 $\{0<u_1<\cdots<u_n<T\}$ 위에서 $n!/T^n$ (같은 점집합을 주는 순열이 $n!$개). $k$번째는 $T\cdot\mathrm{Beta}(k,n-k+1)$, 평균 $kT/(n+1)$.
- **두 가지 시뮬레이션.** (A) 지수 간격의 누적합 → `poisson_arrivals(rate, T, rng)` (정의 2 그대로). (B) $N\sim\mathrm{Poisson}(\lambda T)$를 뽑고 $N$개 균등점을 정렬 → `poisson_arrivals_conditional(n, T, rng)` (정리 2의 결과).

### 2.2 정리 1 — 간격 정의 ⇒ 계수 정의 (Durrett 3e §2.2 Thm 2.1, Ross 12e §5.3.3)

> 정의 2로 만든 $N(t)$는 정의 1을 만족한다. 특히 $N(t)\sim\mathrm{Poisson}(\lambda t)$이고, 증분들은 독립이며 정상이다.

**가정이 왜 필요한가**
- 간격이 **지수분포**여야 한다: 무기억성이 있어야 시각 $s$에서 '다음 도착까지 남은 시간'이 다시 $\mathrm{Exp}(\lambda)$이고 과거와 독립이 되어, 증분의 독립·정상성이 나온다. 간격이 $\mathrm{Gamma}(2)$이면 $N(t)$는 푸아송이 아니다 (E3에서 직접 본다).
- 간격이 **iid**여야 한다: 독립이 아니면 재생 구조가 깨진다.
- $\lambda<\infty$: 유한 시간에 무한 도착이 없도록.

**증명 스케치 (주변분포는 완전히, 증분 구조는 인용).**
(1) *주변분포.* 핵심 항등식은 사건의 동치
$$\{N(t)\ge n\}=\{T_n\le t\}$$
($t$까지 도착이 $n$개 이상 ⟺ $n$번째 도착이 $t$ 이전). $T_n\sim\mathrm{Gamma}(n,\lambda)$의 꼬리 공식(02a 정리 3, Q3) $P(T_n>t)=\sum_{k=0}^{n-1}e^{-\lambda t}(\lambda t)^k/k!$을 쓰면
$$P(N(t)=n)=P(T_n\le t)-P(T_{n+1}\le t)=P(T_{n+1}>t)-P(T_n>t)=e^{-\lambda t}\frac{(\lambda t)^n}{n!}.$$
합에서 $k=n$ 항 하나만 남는 것이 전부다. 이것이 **Gamma–Poisson 쌍대성**이고, 4절에서 두 구성법 모두에서 $E[T_5]=5/\lambda$로 확인한다.
(2) *독립·정상 증분.* 시각 $s$에 조건부로, 다음 도착까지 남은 시간 $T_{N(s)+1}-s$는 무기억성에 의해 다시 $\mathrm{Exp}(\lambda)$이고 $\{N(u),u\le s\}$와 독립이다. 그 뒤의 간격 $\tau_{N(s)+2},\dots$는 원래부터 독립 $\mathrm{Exp}(\lambda)$이므로 $\{N(s+t)-N(s), t\ge0\}$은 $s$ 이전 경로와 독립인 **새 푸아송 과정**이다. 이를 서로소 구간에 차례로 적용하면 (ii), (iii)이 나온다. 엄밀한 서술(정지시간 $N(s)$가 확률변수라는 점의 처리)은 Durrett 3e §2.2 Thm 2.1 / G&S 4e §6.8을 인용한다.

### 2.3 정리 2 — 도착시각의 조건부 분포 = 균등 순서통계량 (Ross 12e §5.3.5 Thm 5.2, Durrett 3e §2.4)

> $N(T)=n$이 주어지면 $(T_1,\dots,T_n)$의 조건부 결합밀도는 $\{0<t_1<\cdots<t_n<T\}$ 위에서 $n!/T^n$, 즉 $n$개 iid $U(0,T)$의 순서통계량과 같다. 따라서 (순서를 무시하면) 도착시각들은 iid $U(0,T)$이며, $E[T_k\mid N(T)=n]=kT/(n+1)$, $E\big[\sum_{i\le N(T)}T_i\big]=\lambda T^2/2$.

**가정이 왜 필요한가**
- 비율이 **상수**여야 균등이다: 비율 $\lambda(t)$가 변하면 조건부 밀도는 $\lambda(t)/\Lambda(T)$가 된다 (02c의 NHPP).
- 간격이 **지수분포**여야 결합밀도가 $\lambda^n e^{-\lambda T}$처럼 $t_1,\dots,t_{n-1}$에 무관해진다 — 이 '평평함'이 균등성의 원천이다. 다른 간격 분포에서는 밀도가 $t_i$의 위치에 의존한다 (E3).

<details><summary>증명 (완전 증명)</summary>

$0<t_1<\cdots<t_n<T$를 고정하고 작은 $dt_1,\dots,dt_n>0$을 잡는다. $t_0=0$이라 두면
$$\{T_i\in(t_i,t_i+dt_i],\ i\le n,\ N(T)=n\}=\{\tau_1\in dt_1,\ \tau_2\in d(t_2-t_1),\ \dots,\ \tau_n\in d(t_n-t_{n-1}),\ \tau_{n+1}>T-t_n\}$$
(오차 $o(\prod dt_i)$). $n$번째 도착까지의 간격을 지정하고, $n+1$번째 간격이 $T$를 넘겨야 $N(T)=n$이 된다. $\tau_i$가 독립 $\mathrm{Exp}(\lambda)$이므로 이 확률은
$$\prod_{i=1}^{n}\lambda e^{-\lambda(t_i-t_{i-1})}\,dt_i\;\cdot\;e^{-\lambda(T-t_n)}
=\lambda^n\exp\!\Big(-\lambda\sum_{i=1}^n(t_i-t_{i-1})-\lambda(T-t_n)\Big)\prod dt_i
=\lambda^n e^{-\lambda T}\prod_{i=1}^n dt_i .$$
지수의 합이 **망원곱(telescoping)** 으로 $-\lambda t_n-\lambda(T-t_n)=-\lambda T$가 되어 $t_1,\dots,t_n$에 전혀 의존하지 않는다. 이제 $P(N(T)=n)=e^{-\lambda T}(\lambda T)^n/n!$ (정리 1)로 나누면 조건부 결합밀도는
$$f(t_1,\dots,t_n\mid N(T)=n)=\frac{\lambda^n e^{-\lambda T}}{e^{-\lambda T}(\lambda T)^n/n!}=\frac{n!}{T^n},\qquad 0<t_1<\cdots<t_n<T .$$
이것은 정확히 $n$개 iid $U(0,T)$의 순서통계량의 결합밀도이다: 정렬되지 않은 $(U_1,\dots,U_n)$의 밀도는 $1/T^n$이고, 같은 정렬 결과를 주는 순열이 $n!$개이므로 정렬된 벡터의 밀도는 $n!/T^n$. $\square$

**따름정리.** (a) 순서를 무작위 순열로 섞으면 $(T_{\pi(1)},\dots,T_{\pi(n)})$은 iid $U(0,T)$. (b) $k$번째 균등 순서통계량은 $T\cdot\mathrm{Beta}(k,n-k+1)$이므로 $E[T_k\mid N(T)=n]=kT/(n+1)$ (E2에서 유도). (c) 합 $\sum_{i\le N(T)}T_i$는 순서에 무관하므로 $E\big[\sum_{i\le n}T_i\mid N(T)=n\big]=n\cdot T/2$, tower로 $E\big[\sum_{i\le N(T)}T_i\big]=E[N(T)]\,T/2=\lambda T^2/2$. (d) 같은 합은 iid $U(0,T)$ 점프를 갖는 복합 푸아송(00c, 02d)이므로 $\mathrm{Var}=\lambda T\,E[U^2]=\lambda T^3/3$.

</details>

### 2.4 정리 3 — 무한소 정의 ⇒ Poisson 분포 (Ross 12e §5.3.2 Thm 5.1)

> 정의 3의 (i)–(iv)로부터 $N(t)\sim\mathrm{Poisson}(\lambda t)$가 따라온다. 즉 정의 3 ⇒ 정의 1.

**가정이 왜 필요한가**
- (iv) $P(N(h)\ge2)=o(h)$가 없으면 한 순간에 여러 개가 몰려 도착하는 과정(복합 푸아송, 02d)도 (iii)을 만족할 수 있다 — 이 조건이 '한 번에 하나(orderliness)'를 강제한다.
- 정상 증분이 없으면 $\lambda$가 시간에 따라 변하는 NHPP(02c)가 된다.

**증명 ($n=0$인 경우만; 나머지는 인용).** $p_n(t)=P(N(t)=n)$이라 하자. (iii)+(iv)에서 $P(N(h)=0)=1-\lambda h+o(h)$. 독립·정상 증분으로
$$p_0(t+h)=P\big(N(t)=0,\ N(t+h)-N(t)=0\big)=p_0(t)\,(1-\lambda h+o(h)),$$
따라서 $\frac{p_0(t+h)-p_0(t)}{h}=-\lambda p_0(t)+o(1)$, 즉 $p_0'=-\lambda p_0$, $p_0(0)=1$ → $p_0(t)=e^{-\lambda t}$. 일반 $n$에 대해서는 같은 논법으로 $p_n'=-\lambda p_n+\lambda p_{n-1}$을 얻고 귀납적으로 풀면 $p_n(t)=e^{-\lambda t}(\lambda t)^n/n!$. 전체 증명은 Ross 12e §5.3.2 Thm 5.1. 4절의 마지막 그림이 (iii), (iv)를 수치로 보여 준다.

### 2.5 직관

세 정의는 같은 대상을 세 눈높이에서 본 것이다. 무한소 정의는 '매 순간 비율 $\lambda$로 동전을 던진다'는 뜻이고(이항 근사 $\mathrm{Bin}(t/h,\lambda h)\to\mathrm{Poisson}(\lambda t)$), 간격 정의는 '동전이 앞면 나올 때까지 기다린 시간이 (기하분포의 극한인) 지수분포'라는 뜻이며, 계수 정의는 그 결과를 요약한다. 순서통계량 성질은 '도착 수만 알면 도착 시각은 완전히 무작위(균등)'라는 말로, 푸아송 과정이 가장 '구조 없는' 점과정임을 보여 준다. 이 성질 덕분에 구성 B(개수 뽑고 균등점 정렬)가 가능하며, 02c의 세분화·중첩·NHPP 증명도 모두 여기서 나온다.

### 2.6 함정

1. '구간 $(s,t]$의 도착 수가 $\mathrm{Poisson}(\lambda(t-s))$'만으로는 푸아송 과정이 아니다 — 증분의 **독립성**이 별도 조건이다 (7절 trap 카드).
2. $T_k$(비조건부)는 $\mathrm{Gamma}(k,\lambda)$이지만, $N(T)=n$에 조건부인 $T_{(k)}$는 $T\cdot\mathrm{Beta}(k,n-k+1)$이다. 4.3절에서 $E[T_5]=2.5$와 $E[T_5\mid N(10)=20]=50/21\approx2.38$이 다름을 본다. 조건부 여부를 항상 명시하라.
3. 구성 B는 $(0,T]$ 위의 과정만 준다. $T$ 이후가 필요하면 새로 만들어야 하고, 구성 A도 마지막 도착 뒤에 '잘린' 간격이 있다는 점에서 같다 (04b의 검사 역설).
4. $P(N(h)=1)=\lambda h\,e^{-\lambda h}$이지 $\lambda h$가 아니다. 차이는 $O(h^2)$이며 $h=0.05$, $\lambda=2$면 이미 $0.1$ vs $0.0905$로 눈에 보인다.
5. 빈(bin) 계수 히스토그램을 그릴 때는 정수 정렬된 빈을 써야 한다(`plot_hist_vs_pmf`가 처리). 임의 빈 폭을 쓰면 pmf 막대와 겹치지 않는다.

✍️ 내 말로: (책을 덮고 여기에 핵심 정의·정리를 다시 써 보세요) — 특히 정의 2에서 정의 1로 가는 길에서 무기억성이 정확히 어디에 쓰이는지 한 문단으로, 그리고 정리 2 증명의 '망원곱' 한 줄을 책 없이 다시 계산해 보세요.

## 3. Predict — 코드를 돌리기 전에 예측

아래 셀의 `None`을 숫자로 바꿔 보세요. 정확할 필요는 없습니다 — 자릿수와 방향이 맞는지가 목적입니다. 예측을 먼저 적어 두고 4절을 실행하세요. ($\lambda=2$, $T=10$)

In [ ]:
predictions = {
    # E[N(T)] 는? (lam=2, T=10)
    "mean_NT": None,
    # P(N(10) = 20) 은? (평균값에서의 pmf — 생각보다 작다)
    "p_NT_eq_20": None,
    # 다섯 번째 도착시각 T_5 의 평균은? (구성 A와 B에서 같아야 한다)
    "mean_T5": None,
    # (0, 10] 안의 모든 도착시각을 더한 sum_{i<=N(10)} T_i 의 평균은?
    "mean_sum_arrivals": None,
    # 폭 h=0.05 인 작은 빈에 도착이 정확히 1개일 확률은? (lam*h = 0.1)
    "p_bin_one": None,
    # 같은 빈에 도착이 2개 이상일 확률은?
    "p_bin_two_plus": None,
}

## 4. Simulate — 시뮬레이션

### 4.1 구성 A — 지수 간격 누적합 (정의 2, 정리 1)

`poisson_arrivals`는 iid $\mathrm{Exp}(\lambda)$ 간격을 누적해 $(0,T]$ 안의 도착시각을 준다. 처음 5개 경로의 계수과정 $N(t)$를 계단 함수로 그리고, 모든 경로의 $N(10)$을 $\mathrm{Poisson}(20)$ pmf와 비교한다.

In [ ]:
paths_A = [poisson_arrivals(LAM, T, rng) for _ in range(N_PATHS)]
N_A = np.array([len(p) for p in paths_A])

t_grid = np.linspace(0, T, 401)
k = np.arange(0, 46)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
for i in range(5):
    ax1.step(t_grid, renewal_count(paths_A[i], t_grid), where="post", lw=1, label=f"path {i+1}")
ax1.plot(t_grid, LAM * t_grid, "k--", lw=1, label="E[N(t)] = lambda t")
ax1.set_xlabel("t"); ax1.set_ylabel("N(t)"); ax1.legend(fontsize=8)
ax1.set_title("Fig 1a: construction A (exponential gaps)")
plot_hist_vs_pmf(N_A, k, stats.poisson(LAM * T).pmf(k), ax=ax2, label_pmf="Poisson(20) pmf")
ax2.set_xlabel("N(10)"); ax2.set_ylabel("density")
ax2.set_title(f"Fig 1b: N(10) over {N_PATHS} paths")
plt.tight_layout(); plt.show()

$\mathrm{Poisson}(\lambda t)$의 평균과 분산은 둘 다 $\lambda t=20$이다. 표본분산의 표준오차는 spkit에 없어 여기서 정의한다: $\mathrm{SE}(s^2)\approx\sqrt{(m_4-m_2^2)/n}$ ($m_k$는 $k$차 중심 표본적률).

In [ ]:
# helper (spkit에 없어서 여기서 정의)
def se_of_variance(x):
    """표본분산의 표준오차 sqrt((m4 - m2^2)/n) (m_k = k차 중심 표본적률)."""
    x = np.asarray(x, dtype=float); n = x.size; d = x - x.mean()
    m2 = (d**2).mean(); m4 = (d**4).mean()
    return float(np.sqrt((m4 - m2**2) / n))

est_meanN = mc_estimate(N_A)
var_N, se_varN = N_A.var(ddof=1), se_of_variance(N_A)
est_p20 = mc_proportion(N_A == 20)
print(f"E[N(10)]    MC {est_meanN}    exact {LAM * T}")
print(f"Var N(10)   sample {var_N:.3f} ± {se_varN:.3f}    exact {LAM * T}")
print(f"P(N(10)=20) MC {est_p20}    exact {stats.poisson(LAM * T).pmf(20):.5f}")

### 4.2 구성 B — 개수를 뽑고 균등점을 정렬 (정리 2)

정리 2가 맞다면 $N\sim\mathrm{Poisson}(\lambda T)$를 뽑은 뒤 $N$개의 $U(0,T)$를 정렬한 것도 같은 과정이다. 두 구성에서 첫 도착 $T_1$과 다섯 번째 도착 $T_5$를 꺼내 분포를 비교한다. $T_5$는 경로에 도착이 5개 이상 있어야 정의되므로 그런 경로만 쓴다 — $P(N(10)<5)$는 $1.7\times10^{-5}$라 이 절단이 만드는 편향($\lesssim2\times10^{-4}$)은 SE보다 두 자릿수 작다.

In [ ]:
Ns = rng.poisson(LAM * T, N_PATHS)
paths_B = [poisson_arrivals_conditional(n, T, rng) for n in Ns]

T5_A = np.array([p[4] for p in paths_A if len(p) >= 5])
T5_B = np.array([p[4] for p in paths_B if len(p) >= 5])
T1_A = np.array([p[0] for p in paths_A if len(p) >= 1])
T1_B = np.array([p[0] for p in paths_B if len(p) >= 1])
print(f"P(N(10) < 5) = {stats.poisson(LAM * T).cdf(4):.2e};  dropped paths: A {N_PATHS - len(T5_A)}, B {N_PATHS - len(T5_B)}")

In [ ]:
x5 = np.linspace(0, 8, 300); x1 = np.linspace(0, 4, 300)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
bins5 = np.linspace(0, 8, 50)
ax1.hist(T5_A, bins=bins5, density=True, alpha=0.45, label="A: exponential gaps")
ax1.hist(T5_B, bins=bins5, density=True, alpha=0.45, label="B: sorted uniforms")
ax1.plot(x5, stats.gamma(a=5, scale=1 / LAM).pdf(x5), "k-", lw=2, label="Gamma(5, rate 2) pdf")
ax1.set_xlabel("T_5"); ax1.set_ylabel("density"); ax1.legend()
ax1.set_title("Both constructions give the same waiting-time law")
bins1 = np.linspace(0, 4, 50)
ax2.hist(T1_A, bins=bins1, density=True, alpha=0.45, label="A: exponential gaps")
ax2.hist(T1_B, bins=bins1, density=True, alpha=0.45, label="B: sorted uniforms")
ax2.plot(x1, stats.expon(scale=1 / LAM).pdf(x1), "k-", lw=2, label="Exp(2) pdf")
ax2.set_xlabel("T_1"); ax2.set_ylabel("density"); ax2.legend()
ax2.set_title("First arrival: min of N uniforms is Exp(2)")
plt.tight_layout(); plt.show()

In [ ]:
est_T5A, est_T5B, est_T1B = mc_estimate(T5_A), mc_estimate(T5_B), mc_estimate(T1_B)
print(f"E[T_5]  A {est_T5A}   B {est_T5B}   exact 5/lambda = {5 / LAM}")
print(f"E[T_1]  B {est_T1B}   exact 1/lambda = {1 / LAM}")

구성 B에서 $T_1$은 '$N$개 균등점의 최솟값'인데도 $\mathrm{Exp}(2)$가 나온다: $P(T_1>t)=E[(1-t/T)^N]=e^{-\lambda T\cdot t/T}=e^{-\lambda t}$ (푸아송 pgf). 정리 2의 역방향이 작동하는 셈이다.

### 4.3 조건부 균등성 (정리 2를 눈으로)

(a) 구성 A의 모든 도착시각을 한데 모으면 밀도가 $1/T$로 평평해야 한다 — 각 경로 안에서 도착시각이 (순서를 무시하면) iid $U(0,T)$이기 때문. (b) 경로별 $\sum_i T_i$의 평균은 $\lambda T^2/2$, 분산은 $\lambda T^3/3$ (따름정리 c, d). (c) $N(10)=20$인 경로만 고르면 $T_5$의 평균이 $5\cdot10/21$로 **내려간다** (함정 2).

In [ ]:
pooled_A = np.concatenate(paths_A)
sums_A = np.array([p.sum() for p in paths_A])
est_sum = mc_estimate(sums_A)
T5_given20 = np.array([p[4] for p in paths_A if len(p) == 20])

fig, ax = plt.subplots()
ax.hist(pooled_A, bins=50, range=(0, T), density=True, alpha=0.5, label="all arrival times, construction A")
ax.axhline(1 / T, color="C1", lw=2, label="1/T = 0.1")
ax.set_ylim(0, 0.13); ax.set_xlabel("arrival time"); ax.set_ylabel("density"); ax.legend()
ax.set_title("Given the count, arrival times are uniform")
plt.show()
print(f"E[sum T_i]    MC {est_sum}    exact lambda T^2/2 = {LAM * T**2 / 2}")
print(f"Var[sum T_i]  sample {sums_A.var(ddof=1):.1f}    exact lambda T^3/3 = {LAM * T**3 / 3:.1f}  (참고)")
print(f"E[T_5 | N(10)=20]  MC {T5_given20.mean():.3f} (n={len(T5_given20)})    exact 5*10/21 = {50 / 21:.4f}   vs unconditional 2.5")

### 4.4 빈 계수 — 독립 Poisson 증분과 무한소 정의 (정의 1, 정의 3)

(a) $[0,10]$을 폭 1인 빈 10개로 나누면 각 빈의 도착 수는 $\mathrm{Poisson}(2)$이고 이웃 빈끼리 독립이어야 한다(상관계수 0). (b) 폭 $h=0.05$인 빈 200개에서는 $P(=1)=\lambda h e^{-\lambda h}$, $P(\ge2)=1-e^{-\lambda h}(1+\lambda h)$이며 그 비는 $\approx\lambda h/2$. (c) $h\to0$에서 $P(=1)/h\to\lambda$, $P(\ge2)/h\to0$ (정의 3의 (iii), (iv)).

In [ ]:
coarse = np.array([count_in_bins(p, T, 10) for p in paths_A])      # (N_PATHS, 10), 폭 1
fine = np.array([count_in_bins(p, T, 200) for p in paths_A])       # (N_PATHS, 200), 폭 0.05
h_fine = T / 200
corr01 = np.corrcoef(coarse[:, 0], coarse[:, 1])[0, 1]
est_p1, est_p2 = mc_proportion(fine == 1), mc_proportion(fine >= 2)
p1_exact = LAM * h_fine * math.exp(-LAM * h_fine)
p2_exact = 1 - math.exp(-LAM * h_fine) * (1 + LAM * h_fine)
print(f"corr(bin 0, bin 1) = {corr01:+.4f}   (SE ≈ 1/sqrt(n) = {1 / np.sqrt(N_PATHS):.4f})")
print(f"h = {h_fine}:  P(=1)  MC {est_p1}   exact {p1_exact:.5f}   (lambda h = {LAM * h_fine})")
print(f"           P(>=2) MC {est_p2}   exact {p2_exact:.5f}")
print(f"           ratio P(>=2)/P(=1) = {est_p2.mean / est_p1.mean:.4f}   ≈ lambda h / 2 = {LAM * h_fine / 2}")

In [ ]:
h_list = [0.5, 0.2, 0.1, 0.05, 0.02]
p1_h, p2_h = [], []
for h in h_list:
    c = np.array([count_in_bins(p, T, int(round(T / h))) for p in paths_A])
    p1_h.append((c == 1).mean() / h); p2_h.append((c >= 2).mean() / h)
hh = np.array(h_list)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
kk = np.arange(0, 11)
plot_hist_vs_pmf(coarse.ravel(), kk, stats.poisson(LAM).pmf(kk), ax=ax1, label_pmf="Poisson(2) pmf")
ax1.set_xlabel("count in a width-1 bin"); ax1.set_ylabel("density"); ax1.set_title("Counts in width-1 bins")
ax2.plot(hh, p1_h, "C0o-", label="P(N(h)=1)/h")
ax2.plot(hh, LAM * np.exp(-LAM * hh), "C0--", lw=1, label="lambda exp(-lambda h)")
ax2.plot(hh, p2_h, "C3s-", label="P(N(h)>=2)/h")
ax2.axhline(LAM, color="k", ls=":", label="lambda = 2"); ax2.axhline(0, color="gray", ls=":")
ax2.set_xscale("log"); ax2.set_xlabel("h (log)"); ax2.set_ylabel("probability / h")
ax2.set_title("Infinitesimal definition"); ax2.legend(fontsize=8)
plt.tight_layout(); plt.show()

$h$가 줄수록 $P(=1)/h$는 $\lambda=2$로, $P(\ge2)/h$는 0으로 간다. 점선 $\lambda e^{-\lambda h}$가 보여 주듯 $P(=1)/h$의 $\lambda$와의 차이는 $O(h)$ (함정 4).

## 5. Compare — 예측 · 시뮬레이션 · 닫힌 형식 비교

In [ ]:
p20_exact = stats.poisson(LAM * T).pmf(20)
rows = [
    {"name": "E[N(10)]", "predicted": predictions["mean_NT"], "estimate": est_meanN, "exact": LAM * T},
    {"name": "Var N(10)", "predicted": None, "estimate": var_N, "se": se_varN, "exact": LAM * T},
    {"name": "P(N(10) = 20)", "predicted": predictions["p_NT_eq_20"], "estimate": est_p20, "exact": p20_exact},
    {"name": "E[T_5], construction A", "predicted": predictions["mean_T5"], "estimate": est_T5A, "exact": 5 / LAM},
    {"name": "E[T_5], construction B", "predicted": predictions["mean_T5"], "estimate": est_T5B, "exact": 5 / LAM},
    {"name": "E[T_1], construction B", "predicted": None, "estimate": est_T1B, "exact": 1 / LAM},
    {"name": "E[sum of arrival times]", "predicted": predictions["mean_sum_arrivals"], "estimate": est_sum, "exact": LAM * T**2 / 2},
    {"name": "P(N(h) = 1), h = 0.05", "predicted": predictions["p_bin_one"], "estimate": est_p1, "exact": p1_exact},
    {"name": "P(N(h) >= 2), h = 0.05", "predicted": predictions["p_bin_two_plus"], "estimate": est_p2, "exact": p2_exact},
    {"name": "corr(adjacent width-1 bins)", "predicted": None, "estimate": corr01, "se": 1 / np.sqrt(N_PATHS), "exact": 0.0},
]
Markdown(compare_table(rows))

행별로 '불일치'가 뜻하는 것:
- **E[N(10)]**: 어긋나면 `poisson_arrivals`의 간격 스케일(`1/rate`)이나 $(0,T]$ 절단이 틀린 것이다.
- **Var N(10)**: 평균은 맞는데 분산이 다르면 간격이 지수분포가 아니거나(E3) 독립이 아닌 것이다 — 분산/평균 비 1이 푸아송의 지문이다.
- **P(N(10) = 20)**: 어긋나면 정리 1의 Gamma–Poisson 쌍대성(주변분포)이 깨진 것이다.
- **E[T_5] (A)**: 어긋나면 $T_5\sim\mathrm{Gamma}(5,\lambda)$, 즉 지수 합의 분포가 틀린 것이다.
- **E[T_5] (B)**: A는 맞는데 B만 어긋나면 정리 2(조건부 균등 순서통계량)가 틀렸거나 `poisson_arrivals_conditional`의 정렬이 빠진 것이다.
- **E[T_1] (B)**: 어긋나면 '$N$개 균등점의 최솟값이 $\mathrm{Exp}(\lambda)$'라는 pgf 계산이 틀린 것이다.
- **E[sum of arrival times]**: 어긋나면 tower 계산 $E[N]\cdot T/2$, 즉 조건부 균등성이 깨진 것이다.
- **P(N(h) = 1)**: 어긋나면 작은 빈의 계수가 $\mathrm{Poisson}(\lambda h)$가 아닌 것이고, $\lambda h$(근사)로 비교하면 3 SE 이상 어긋나는 것이 정상이다 (함정 4).
- **P(N(h) >= 2)**: 어긋나면 한 순간에 여러 도착이 몰리는 구조(정의 3의 (iv) 위반)가 있는 것이다.
- **corr(adjacent bins)**: 0에서 벗어나면 독립 증분이 깨진 것이다 (E3의 재생과정은 여기서 음의 상관을 보인다).

In [ ]:
assert_close(est_meanN, LAM * T, k=4, name="mean_NT")
assert_close(var_N, LAM * T, se=se_varN, k=4, name="var_NT")
assert_close(est_p20, p20_exact, k=4, name="p_NT_eq_20")
assert_close(est_T5A, 5 / LAM, k=4, name="mean_T5_A")
assert_close(est_T5B, 5 / LAM, k=4, name="mean_T5_B")
assert_close(est_T1B, 1 / LAM, k=4, name="mean_T1_B")
assert_close(est_sum, LAM * T**2 / 2, k=4, name="mean_sum_arrivals")
assert_close(est_p1, p1_exact, k=4, name="p_bin_one")
assert_close(est_p2, p2_exact, k=4, name="p_bin_two_plus")
assert_close(corr01, 0.0, se=1 / np.sqrt(N_PATHS), k=4, name="corr_adjacent_bins")
print("all checks passed")

## 6. 연습문제

### E1 계산

$\lambda=2$인 푸아송 과정. 손으로 구한 뒤 코드로 확인하라.
(a) $N(10)=20$이 주어졌을 때 $P(N(3)=2\mid N(10)=20)$.
(b) 비조건부로 $P(N(3)=2,\ N(10)=20)$.
(c) $E[T_3\mid N(10)=20]$.

In [ ]:
# 여기에 풀이

<details><summary>정답</summary>

(a) 조건부 균등성(정리 2): 20개 점 각각이 독립적으로 $(0,3]$에 떨어질 확률은 $3/10=0.3$이므로 $N(3)\mid N(10)=20\sim\mathrm{Bin}(20,0.3)$:
$$P=\binom{20}{2}0.3^2\,0.7^{18}\approx0.02785 .$$
(b) 독립 증분(정의 1): $\{N(3)=2,\ N(10)=20\}=\{N(3)=2\}\cap\{N(10)-N(3)=18\}$이고 두 증분은 독립 $\mathrm{Poisson}(6)$, $\mathrm{Poisson}(14)$:
$$P=e^{-6}\frac{6^2}{2!}\cdot e^{-14}\frac{14^{18}}{18!}\approx0.002474 .$$
검산: (b)/(a) $=P(N(10)=20)=e^{-20}20^{20}/20!\approx0.0888$ — 5절의 값과 같다.
(c) $E[T_k\mid N(T)=n]=kT/(n+1)=3\cdot10/21=10/7\approx1.4286$.

```python
a = stats.binom(20, 0.3).pmf(2)
b = stats.poisson(6).pmf(2) * stats.poisson(14).pmf(18)
c = 3 * 10 / 21
print(f"(a) {a:.6f}  (b) {b:.7f}  (c) {c:.5f}   check (b)/(a) = {b / a:.5f} = P(N(10)=20)")
# (a) 0.027846  (b) 0.0024737  (c) 1.42857   check 0.08884
```

</details>

### E2 유도 후 시뮬레이션 검증

정리 2로부터 $N(T)=n$일 때 $T_{(k)}\sim T\cdot\mathrm{Beta}(k,n-k+1)$, 특히 $E[T_{(k)}\mid N(T)=n]=kT/(n+1)$임을 유도하라. 그런 다음 $n=20$, $T=10$, $k=5$로 시뮬레이션해 `assert_close`로 검증하라 (분산 $T^2\frac{k(n-k+1)}{(n+1)^2(n+2)}$도 참고로 비교).

In [ ]:
# 여기에 풀이

<details><summary>정답</summary>

**유도.** $n$개 iid $U(0,T)$ 중 $k$번째 순서통계량이 $(u,u+du]$에 있으려면: 하나가 그 안에($n$가지 선택, 확률 $du/T$), 나머지 중 $k-1$개가 $u$ 아래($\binom{n-1}{k-1}$가지, 각 $u/T$), 나머지 $n-k$개가 $u$ 위(각 $(T-u)/T$). 따라서
$$f_{(k)}(u)=\frac{n!}{(k-1)!\,(n-k)!}\,\frac{u^{k-1}(T-u)^{n-k}}{T^n},\qquad 0<u<T,$$
이는 $u/T\sim\mathrm{Beta}(k,n-k+1)$의 밀도이다. Beta 평균 $\frac{k}{k+(n-k+1)}=\frac{k}{n+1}$이므로 $E[T_{(k)}\mid N(T)=n]=\frac{kT}{n+1}$, 분산은 $T^2\frac{k(n-k+1)}{(n+1)^2(n+2)}$. $n=20,k=5,T=10$: 평균 $50/21\approx2.3810$, 분산 $100\cdot\frac{5\cdot16}{21^2\cdot22}\approx0.8246$.

```python
n_rep = scale(50_000)
Ts = np.array([poisson_arrivals_conditional(20, 10.0, rng)[4] for _ in range(n_rep)])
est = mc_estimate(Ts)
var_exact = 100 * stats.beta(5, 16).var()
print(f"E[T_(5) | N=20]  MC {est}   exact 50/21 = {50 / 21:.5f}")
print(f"Var              sample {Ts.var(ddof=1):.4f}   exact {var_exact:.4f}")
assert_close(est, 50 / 21, k=4, name="E2 conditional T_(5)")
# full 모드 SE ≈ 0.004, FAST ≈ 0.013; 4.3절의 (n=N(10)=20 경로만 고른) 추정과 일치
```

</details>

### E3 가정을 깨보기

간격을 $\mathrm{Exp}(2)$ 대신 **평균이 같은** $\mathrm{Gamma}(2,\text{rate}=4)$ (평균 $0.5$, 분산 $1/8$)로 바꾼 재생과정을 만들어라 (`renewal_times(lambda n, g: g.gamma(2.0, 0.25, n), T, rng)`; numpy의 두 번째 인자는 척도 $1/4$).
(a) $N(10)$의 평균과 분산, 분산/평균 비(dispersion index)를 구하라. 푸아송이면 1이다.
(b) 폭 1인 빈 계수 히스토그램을 $\mathrm{Poisson}(2)$ pmf와 비교하고, 이웃 빈의 상관계수를 구하라.
(c) 모든 도착시각을 모은 히스토그램이 여전히 평평한가? 무엇이 달라지는가?

In [ ]:
# 여기에 풀이

<details><summary>정답</summary>

(a) 재생 정리(04a)의 점근식 $\mathrm{Var}\,N(t)\approx\sigma^2t/\mu^3=(1/8)\cdot10/(1/8)=10$, $E[N(t)]\approx t/\mu=20$ (실제로는 약 19.7 — 첫 도착까지 지수분포보다 늦게 시작하는 초기 편향). 따라서 dispersion index $\approx\sigma^2/\mu^2=0.5$. MC(20k 경로) 확인값: 평균 $19.7$, 분산 $\approx10$, 비 $\approx0.5$.
(b) 빈 계수는 $\mathrm{Poisson}(2)$보다 2 근처에 몰려 **뾰족**하다(과소산포, underdispersion). 이웃 빈의 상관은 약간 **음수**: 한 빈에 많이 왔으면 간격이 짧았다는 뜻이고, Gamma(2) 간격은 지수보다 규칙적이라 다음 빈은 상대적으로 적다.
(c) 풀링 히스토그램은 초반($t\lesssim1$)에 뚜렷이 낮고 그 뒤로는 평평해진다. 정상 증분이 아니어서(시각 0에서 '새 간격이 막 시작'하는 것이 특별한 상태) 시작 근처에 구조가 생긴다. 요점: 간격이 규칙적일수록 계수의 분산이 줄고, 푸아송은 '완전 무작위'의 기준점(비 1)이다.

```python
n_e3 = scale(20_000)
paths_G = [renewal_times(lambda n, g: g.gamma(2.0, 0.25, n), T, rng) for _ in range(n_e3)]
N_G = np.array([len(p) for p in paths_G])
coarse_G = np.array([count_in_bins(p, T, 10) for p in paths_G])
pooled_G = np.concatenate(paths_G)
print(f"(a) E[N(10)] {N_G.mean():.3f}   Var {N_G.var(ddof=1):.3f}   dispersion {N_G.var(ddof=1) / N_G.mean():.3f}   (Poisson: 1)")
print(f"(b) corr(bin 0, bin 1) = {np.corrcoef(coarse_G[:, 0], coarse_G[:, 1])[0, 1]:+.4f}")
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
kk = np.arange(0, 11)
plot_hist_vs_pmf(coarse_G.ravel(), kk, stats.poisson(2).pmf(kk), ax=ax1, label_pmf="Poisson(2) pmf")
ax1.set_title("Gamma(2, rate 4) renewal: width-1 bin counts (underdispersed)")
ax2.hist(pooled_G, bins=50, range=(0, T), density=True, alpha=0.5, label="pooled arrival times")
ax2.axhline(0.1, color="C1", lw=2, label="1/T"); ax2.set_ylim(0, 0.13); ax2.legend()
ax2.set_title("Not uniform near t = 0 (no stationary increments)")
plt.tight_layout(); plt.show()
# (a) 약 19.7 / 10.0 / 0.50   (b) 약 -0.05
```

</details>

## 7. 정리

1. 세 정의(증분/간격/무한소)는 동치이며, 간격 정의가 시뮬레이션(구성 A), 증분 정의가 분석, 무한소 정의가 모형화(비율 = 순간 도착 확률/시간)에 쓰인다.
2. $P(N(t)\ge n)=P(T_n\le t)$: 계수와 대기시간은 같은 사건의 두 표현이고, 이 Gamma–Poisson 쌍대성이 $N(t)\sim\mathrm{Poisson}(\lambda t)$를 준다.
3. $N(T)=n$이 주어지면 도착시각은 iid $U(0,T)$의 순서통계량 — 결합밀도가 망원곱으로 $\lambda^ne^{-\lambda T}$로 평평해지기 때문. 이것이 구성 B의 근거이자 02c의 세분화·NHPP 증명의 열쇠다.
4. 구간 계수는 독립 $\mathrm{Poisson}(\lambda h)$이고, 작은 $h$에서 $P(1)\approx\lambda h$, $P(\ge2)\approx(\lambda h)^2/2$.
5. 간격이 지수분포가 아니면(E3) 계수는 푸아송이 아니고 분산/평균 비가 1에서 벗어난다 — 푸아송은 '완전 무작위' 점과정의 기준점이다.

**trap 카드.**
Q: '각 구간의 도착 수가 $\mathrm{Poisson}(\lambda\cdot\text{길이})$이다'라고만 하면 푸아송 과정인가?
A: 아니다. 증분의 **독립성**이 빠졌다. 예: $M\sim\mathrm{Poisson}(\lambda)$ 하나를 뽑아 $N(t)=\lfloor Mt\rfloor$로 두면 $N(1)=M$은 $\mathrm{Poisson}(\lambda)$이지만 서로소 구간의 증분이 $M$ 하나로 완전히 결정되어 종속이다. 정의 1의 (ii)와 (iii)은 둘 다 필요하다.

**다음 노트북: 02c (세분화·중첩·NHPP).** 정리 2(조건부 균등성)와 정의 1을 손에 쥐면 '점마다 독립 동전을 던져 나누기(세분화)', '두 과정 합치기(중첩)', '비율이 시간에 따라 변하기(NHPP)'가 모두 몇 줄 증명으로 나온다. 02c는 그 세 변환을 다룬다.

log/progress.md 한 줄 템플릿: `- [ ] 02b 완료 (YYYY-MM-DD) · Predict 적중 __/6 · E1 __ E2 __ E3 __ · 막힌 곳: __`